# NG IBKR Hybrid Loader

Loads natural gas bars from the IBKR Client Portal API and reconstructs the `HybridMixtureNetwork` using the best trial from `notebooks/ng_hybrid_intraday_optuna.ipynb`.

This notebook does three things:
1. Pull recent NG history from IBKR for `conid=269460054`.
2. Rebuild the best hybrid model configuration from Optuna trial `#16`.
3. Load the best checkpoint weights from `ng_hybrid_intraday_best.pth` when available.

The training notebook selected:
- Trial: `#16`
- Score: `17.1104`
- `n_features=55`, `seq_len=20`, `ae_window_bars=510`
- Session: `USA 02:30-15:00`
- Bar size: `15min`
- Horizon: `10` bars


In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import torch
from IPython.display import display

from CTAFlow.data.ext.ibkr_client import IBKRConfig, IBKRContract, IBKRTickDataSource
from CTAFlow.models.deep_learning.multi_branch.ng_moe import HybridConfig, HybridMixtureNetwork

warnings.filterwarnings(
    "ignore",
    message="Unverified HTTPS request",
)

pd.set_option("display.max_columns", 200)
torch.set_grad_enabled(False)


In [ ]:
# IBKR connection
GATEWAY_URL = "https://localhost:5000"
CONID = "269460054"
ACCOUNT_ID = ""
VERIFY_SSL = False
TZ = "America/Chicago"
HISTORY_PERIOD = "5d"
BAR_SIZE = "15min"

# Optional override if your checkpoint lives elsewhere.
CHECKPOINT_PATH = None
CHECKPOINT_CANDIDATES = [
    Path("/workspace/results/ng_hybrid_intraday/ng_hybrid_intraday_best.pth"),
    Path("results/ng_hybrid_intraday/ng_hybrid_intraday_best.pth"),
    Path("ng_hybrid_intraday_best.pth"),
]

# Best selected trial from notebooks/ng_hybrid_intraday_optuna.ipynb.
BEST_PARAMS = {
    "trial_number": 16,
    "best_value": 17.1104,
    "dropout": 0.10533235048468705,
    "mdn_n_components": 4,
    "vsn_d_model": 16,
    "vsn_temperature": 1.7848776002817843,
    "vsn_entropy_weight": 0.04634773780446447,
    "vsn_min_weight": 0.04685948948098972,
    "ce_weight": 1.1161458818666021,
    "positioning_pnl_weight": 0.3532400556872766,
    "nll_weight": 0.09233749107554323,
    "kl_weight": 0.013191355146703608,
    "recon_weight": 0.1512916773328667,
    "tc_cost": 0.0009520881533881362,
    "lr": 0.00017517345199822215,
    "weight_decay": 0.00033058060193960235,
}

LOCKED_ARCH = {
    "d_latent": 32,
    "d_ae_hidden": 256,
    "tcn_width": 64,
    "tcn_depth": 4,
    "stride": 2,
    "mdn_hidden": 128,
    "head_hidden_dim": 256,
    "positioning_hidden_dim": 64,
    "batch_size": 128,
}

TRAINING_META = {
    "n_features": 55,
    "seq_len": 20,
    "f_ae": 12,
    "ae_window": 510,
    "bar_minutes": 15,
    "target_horizon_bars": 10,
    "n_classes": 4,
    "use_positioning_head": True,
    "use_vsn": True,
    "session": "USA_02:30-15:00",
}

FEATURE_GROUP_SIZES = {
    "technical": 34,
    "storage": 8,
    "weather": 13,
}


def resolve_checkpoint_path() -> Path | None:
    if CHECKPOINT_PATH:
        path = Path(CHECKPOINT_PATH).expanduser()
        return path if path.exists() else path
    for candidate in CHECKPOINT_CANDIDATES:
        if candidate.exists():
            return candidate
    return None


def build_best_config() -> HybridConfig:
    return HybridConfig(
        n_features=TRAINING_META["n_features"],
        seq_len=TRAINING_META["seq_len"],
        f_ae=TRAINING_META["f_ae"],
        ae_window=TRAINING_META["ae_window"],
        d_latent=LOCKED_ARCH["d_latent"],
        d_ae_hidden=LOCKED_ARCH["d_ae_hidden"],
        kl_weight=BEST_PARAMS["kl_weight"],
        recon_weight=BEST_PARAMS["recon_weight"],
        tcn_channels=[LOCKED_ARCH["tcn_width"]] * LOCKED_ARCH["tcn_depth"],
        tcn_kernel_size=3,
        stride=LOCKED_ARCH["stride"],
        mdn_hidden_dims=[LOCKED_ARCH["mdn_hidden"], LOCKED_ARCH["mdn_hidden"] // 2],
        mdn_n_components=BEST_PARAMS["mdn_n_components"],
        head_hidden_dim=LOCKED_ARCH["head_hidden_dim"],
        dropout=BEST_PARAMS["dropout"],
        nll_weight=BEST_PARAMS["nll_weight"],
        n_classes=TRAINING_META["n_classes"],
        use_positioning_head=TRAINING_META["use_positioning_head"],
        positioning_hidden_dim=LOCKED_ARCH["positioning_hidden_dim"],
        ce_weight=BEST_PARAMS["ce_weight"],
        positioning_pnl_weight=BEST_PARAMS["positioning_pnl_weight"],
        tc_cost=BEST_PARAMS["tc_cost"],
        use_vsn=TRAINING_META["use_vsn"],
        vsn_d_model=BEST_PARAMS["vsn_d_model"],
        vsn_temperature=BEST_PARAMS["vsn_temperature"],
        vsn_entropy_weight=BEST_PARAMS["vsn_entropy_weight"],
        vsn_min_weight=BEST_PARAMS["vsn_min_weight"],
    )


best_params_frame = pd.Series({**BEST_PARAMS, **LOCKED_ARCH, **TRAINING_META}).to_frame("value")
display(best_params_frame)
resolved_checkpoint = resolve_checkpoint_path()
print(f"Resolved checkpoint: {resolved_checkpoint}")


In [ ]:
ibkr = IBKRTickDataSource(
    config=IBKRConfig(
        base_url=GATEWAY_URL,
        account_id=ACCOUNT_ID,
        verify_ssl=VERIFY_SSL,
        timeout=15.0,
    ),
    contracts={"NG": IBKRContract(conid=CONID, ticker="NG", bar_size=BAR_SIZE)},
    tz=TZ,
)

tickle = ibkr.tickle()
snapshot = ibkr.get_snapshot(CONID)
history = ibkr.get_history(
    conid=CONID,
    period=HISTORY_PERIOD,
    bar_size=BAR_SIZE,
    outside_rth=True,
)

if history.empty:
    raise RuntimeError("IBKR returned no NG history.")

history_local = history.copy()
history_local["Datetime"] = pd.to_datetime(history_local["Datetime"], utc=True).dt.tz_convert(TZ)
history_local = history_local.set_index("Datetime").sort_index()

display(pd.Series({
    "tickle_keys": list(tickle.keys()) if isinstance(tickle, dict) else type(tickle).__name__,
    "snapshot_type": type(snapshot).__name__,
    "bars": len(history_local),
    "start": history_local.index.min(),
    "end": history_local.index.max(),
})).to_frame("value")
display(history_local.tail(10))
display(pd.DataFrame(snapshot if isinstance(snapshot, list) else [snapshot]).head())


In [ ]:
checkpoint = None
if resolved_checkpoint is not None and resolved_checkpoint.exists():
    checkpoint = torch.load(resolved_checkpoint, map_location="cpu", weights_only=False)
    print(f"Loaded checkpoint from {resolved_checkpoint}")
else:
    print("Checkpoint not found. Set CHECKPOINT_PATH to ng_hybrid_intraday_best.pth before running this cell again.")

if checkpoint is not None and "config" in checkpoint:
    model_cfg = HybridConfig(**checkpoint["config"])
else:
    model_cfg = build_best_config()

model = HybridMixtureNetwork(
    model_cfg,
    feature_group_sizes=FEATURE_GROUP_SIZES if model_cfg.use_vsn else None,
)

if checkpoint is not None and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

summary = {
    "parameters": sum(p.numel() for p in model.parameters()),
    "n_features": model_cfg.n_features,
    "seq_len": model_cfg.seq_len,
    "ae_window": model_cfg.ae_window,
    "use_vsn": model_cfg.use_vsn,
    "n_classes": model_cfg.n_classes,
    "checkpoint_loaded": checkpoint is not None,
}
display(pd.Series(summary).to_frame("value"))

if checkpoint is not None:
    for key in [
        "best_params",
        "bar_minutes",
        "target_horizon_bars",
        "session",
        "feature_cols",
        "feature_groups",
        "regime_cols",
        "ae_window_bars",
    ]:
        if key in checkpoint:
            value = checkpoint[key]
            if isinstance(value, list) and len(value) > 10:
                print(f"{key}: {len(value)} entries")
            else:
                print(f"{key}: {value}")


In [ ]:
# Optional: simple raw-bar visualization and latest close summary.
ax = history_local["Close"].plot(figsize=(12, 4), title="IBKR NG close history")
ax.set_ylabel("Close")

latest_bar = history_local.iloc[-1]
latest_summary = pd.Series({
    "latest_ts": history_local.index[-1],
    "open": float(latest_bar["Open"]),
    "high": float(latest_bar["High"]),
    "low": float(latest_bar["Low"]),
    "close": float(latest_bar["Close"]),
    "volume": float(latest_bar["TotalVolume"]),
})
display(latest_summary.to_frame("value"))

print(
    "Model and IBKR data are loaded. "
    "For full live inference, the next step is to build training-aligned 55-feature tensors "
    "from intraday bars plus storage/weather context, or to point the live runner at the saved checkpoint and daily caches."
)
